# Delineate shapfile using web-based API and save as GeoJSON files.

In [19]:
import os
import time
import requests
import pandas as pd
import json

# 1. Load CSV
csv_file = "rawdata/selected_hydro_stations.csv"
df = pd.read_csv(csv_file) # expects columns: Lat, Lon, Station_ID

# 2. Output folder
output_folder = "rawdata/web_based"
os.makedirs(output_folder, exist_ok=True)

# 3. ✅  API endpoint
API_URL = "https://mghydro.com/app/watershed_api"

print(f"Loaded {len(df)} points. Starting delineation...\n")

for i, row in df.iterrows():

    lat = float(row["lat"])
    lng = float(row["lon"])   # IMPORTANT: API uses 'lng', not 'lon'

    name = row.get("station", f"station_{i}")
    point_id = row.get("Name", f"basin_{name}")

    output_file = os.path.join(output_folder, f"{point_id}.geojson")

    print(f"[{i+1}/{len(df)}] Processing {point_id} ({lat}, {lng})")

    params = {
        "lat": lat,
        "lng": lng,
        "precision": "high",   # optional but better quality
        "simplify": "true" # optional but smaller file size
    }

    try:
        r = requests.get(API_URL, params=params, timeout=120)

        print("   Status:", r.status_code)

        if r.status_code != 200:
            print("   ❌ Error:", r.text[:300])
            continue

        # Must be GeoJSON
        data = r.json()

        with open(output_file, "w") as f:
            json.dump(data, f)

        print(f"   ✅ Saved -> {output_file}")

    except Exception as e:
        print("   ❌ Failed:", e)

    time.sleep(3)  # important (API rate limit warning)

Loaded 15 points. Starting delineation...

[1/15] Processing basin_120.0 (29.6722, 80.5583)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_120.0.geojson
[2/15] Processing basin_259.2 (29.3, 80.775)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_259.2.geojson
[3/15] Processing basin_260.0 (28.9778, 81.1444)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_260.0.geojson
[4/15] Processing basin_289.95 (28.3511, 81.7206)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_289.95.geojson
[5/15] Processing basin_375.0 (28.0006, 82.1161)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_375.0.geojson
[6/15] Processing basin_406.5 (28.2542, 83.7242)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_406.5.geojson
[7/15] Processing basin_438.0 (28.1, 84.2333)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_438.0.geojson
[8/15] Processing basin_439.7 (27.95, 84.43)
   Status: 200
   ✅ Saved -> rawdata/web_based\basin_439.7.geojson
[9/15] Processing basin_445.0 (28.0436

# For all delineated catchment, calculate the drainage area in km2 and save to a csv file.

In [20]:
import os
import geopandas as gpd
import pandas as pd

geojson_folder = "rawdata/web_based"
output_folder = 'rawdata/web_based'

results = []

print("Computing catchment areas...\n")

for file in os.listdir(geojson_folder):
    if not file.endswith(".geojson"):
        continue

    path = os.path.join(geojson_folder, file)

    try:
        gdf = gpd.read_file(path)

        # merge geometry in case of multiple features
        geom = gdf.geometry.unary_union

        basin_id = file.replace(".geojson", "")

        # convert to GeoDataFrame
        gdf_single = gpd.GeoDataFrame([1], geometry=[geom], crs="EPSG:4326")

        # project to equal-area CRS
        gdf_proj = gdf_single.to_crs(epsg=6933)

        area_m2 = gdf_proj.geometry.area.values[0]
        area_km2 = area_m2 / 1e6

        results.append({
            "basin_id": basin_id,
            "area_km2": area_km2
        })

        # print(f"{basin_id}: {area_km2:.2f} km²")

    except Exception as e:
        print(f"Failed {file}: {e}")

# final dataframe
df = pd.DataFrame(results)
# round to 0 decimal
df["area_km2"] = df["area_km2"].round(0)
# sort by area low to high
df = df.sort_values("area_km2").reset_index(drop=True)
# print if area is greater than 1000 km²
print("\nCatchment Areas (sorted by size):")
print(df[df["area_km2"] <= 10000])

df.to_csv(f"{output_folder}/catchment_areas.csv", index=False)

print("\nSaved: catchment_areas.csv")

Computing catchment areas...


Catchment Areas (sorted by size):
        basin_id  area_km2
0    basin_406.5     565.0
1    basin_438.0     852.0
2    basin_120.0    1186.0
3    basin_610.0    2363.0
4   basin_289.95    2598.0
5    basin_589.0    2805.0
6    basin_670.0    3716.0
7    basin_445.0    3874.0
8    basin_439.7    4047.0
9    basin_259.2    4307.0
10   basin_447.0    4625.0
11   basin_630.0    4842.0
12   basin_375.0    5290.0
13   basin_690.0    5884.0
14   basin_260.0    7356.0

Saved: catchment_areas.csv


C:\Users\sp2596\AppData\Local\Temp\ipykernel_5856\2861120138.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geometry.unary_union
C:\Users\sp2596\AppData\Local\Temp\ipykernel_5856\2861120138.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geometry.unary_union
C:\Users\sp2596\AppData\Local\Temp\ipykernel_5856\2861120138.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geometry.unary_union
C:\Users\sp2596\AppData\Local\Temp\ipykernel_5856\2861120138.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geometry.unary_union
C:\Users\sp2596\AppData\Local\Temp\ipykernel_5856\2861120138.py:22: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = gdf.geomet